In [0]:
%pip install tensorflow

In [0]:
import os
import numpy as np
import pandas as pd
import joblib
import pickle

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.linear_model import LinearRegression

In [0]:
# =========================
# Paths
# =========================
model_path = "/Volumes/workspace/default/ather/models/"
pred_path = "/Volumes/workspace/default/ather/predictions/"

os.makedirs(model_path, exist_ok=True)
os.makedirs(pred_path, exist_ok=True)

In [0]:
# =========================
# Load LSTM Data
# =========================
X_train_lstm = np.load("/Volumes/workspace/default/ather/data/X_train_lstm.npy")
y_train_lstm = np.load("/Volumes/workspace/default/ather/data/y_train_lstm.npy")

X_test_lstm = np.load("/Volumes/workspace/default/ather/data/X_test_lstm.npy")
y_test_lstm = np.load("/Volumes/workspace/default/ather/data/y_test_lstm.npy")

In [0]:
# =========================
# Build LSTM Model
# =========================
model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])),
    Dropout(0.2),
    LSTM(64),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

In [0]:
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

In [0]:
# =========================
# Train LSTM
# =========================
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [0]:
model.fit(
    X_train_lstm,
    y_train_lstm,
    epochs=20,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

In [0]:
# =========================
# LSTM Predictions
# =========================
y_pred_lstm = model.predict(X_test_lstm).flatten()

# Align shapes with traditional models
min_len = min(len(y_pred_lstm), len(y_test_lstm))
y_pred_lstm = y_pred_lstm[:min_len]
y_test_lstm = y_test_lstm[:min_len]

In [0]:
# =========================
# Save LSTM Model
# =========================
import shutil
import tempfile

# Save to temporary local path first
temp_dir = tempfile.mkdtemp()
temp_model_path = os.path.join(temp_dir, "lstm_model.keras")
model.save(temp_model_path)

# Copy to Volume
final_path = model_path + "lstm_model.keras"
shutil.copy(temp_model_path, final_path)

# Clean up temp file
shutil.rmtree(temp_dir)

In [0]:
# =========================
# Load Traditional Model Predictions
# =========================
pred_df = pd.read_csv(pred_path + "predictions.csv")

# Align lengths
min_len_all = min(len(pred_df), len(y_pred_lstm))
pred_df = pred_df.iloc[:min_len_all]

In [0]:
# =========================
# Select Best Models (example: RF + XGB + LSTM)
# =========================
ensemble_features = pd.DataFrame({
    "rf": pred_df["RandomForest"].values[:min_len_all],
    "xgb": pred_df["XGBoost"].values[:min_len_all],
    "lstm": y_pred_lstm[:min_len_all]
})

y_true = pred_df["y_true"].values[:min_len_all]

In [0]:
# =========================
# Train Ensemble Model (Meta Learner)
# =========================
ensemble_model = LinearRegression()
ensemble_model.fit(ensemble_features, y_true)

In [0]:
# =========================
# Ensemble Predictions
# =========================
y_pred_ensemble = ensemble_model.predict(ensemble_features)

In [0]:
# =========================
# Save Ensemble Model
# =========================
joblib.dump(ensemble_model, model_path + "ensemble_model.pkl")

In [0]:
# =========================
# Save Predictions
# =========================
final_pred_df = pd.DataFrame({
    "y_true": y_true,
    "LSTM": y_pred_lstm[:min_len_all],
    "Ensemble": y_pred_ensemble
})

final_pred_df.to_csv(pred_path + "lstm_ensemble_predictions.csv", index=False)

with open(pred_path + "lstm_ensemble_predictions.pkl", "wb") as f:
    pickle.dump(final_pred_df, f)